# 05 — Evaluation & IEEE Figures

Loads both trained checkpoints from Drive, computes test-set metrics, and produces all IEEE-quality figures (PNG + PDF) in `results/plots/` and `report/figures/`.

Run notebooks 03 and 04 first.

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## 2. Config — pick the two trained models

In [ ]:
MODELS = ["resnet50", "efficientnet_b2"]    # match what you trained in 03 + 04
BATCH_SIZE = 32

## 3. Imports + data loaders

In [ ]:
import torch
from config.config import (
    CLASS_NAMES, DATASET_DIR, CHECKPOINTS_DRIVE,
    LOCAL_PLOTS, REPORT_FIGURES, LOCAL_METRICS, RESULTS_DRIVE, LOGS_DRIVE,
)
from src.utils import get_device, set_seed, read_json
from src.dataset import build_dataloaders
from src.models import build_model
from src.train import load_model_for_inference
from src.evaluate import evaluate_and_save, plot_model_comparison

set_seed(42)
device = get_device()
_, _, test_loader, class_to_idx = build_dataloaders(
    dataset_root=DATASET_DIR, batch_size=BATCH_SIZE,
)
print(f"Test set: {len(test_loader.dataset)} images, {len(test_loader)} batches")

## 4. Per-model evaluation + plots

In [ ]:
metrics_by_model = {}

for model_name in MODELS:
    print(f"\n=== {model_name} ===")
    ckpt = CHECKPOINTS_DRIVE / f"{model_name}_best.pth"
    if not ckpt.exists():
        print(f"  ⚠️ skipping — checkpoint not found: {ckpt}")
        continue
    model, _ = build_model(model_name)
    model = load_model_for_inference(model, ckpt, device)

    # Pull training history from Drive if it exists, so we can draw curves
    history_path = LOGS_DRIVE / f"{model_name}_history.json"
    history = read_json(history_path) if history_path.exists() else None

    m = evaluate_and_save(
        model=model,
        test_loader=test_loader,
        device=device,
        class_names=CLASS_NAMES,
        model_name=model_name,
        plots_dir=LOCAL_PLOTS,
        report_dir=REPORT_FIGURES,
        metrics_dir=LOCAL_METRICS,
        history=history,
    )
    metrics_by_model[model_name] = m
    print(f"  accuracy: {m['accuracy']:.4f}  macro F1: {m['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 5. Cross-model comparison figure

In [ ]:
if len(metrics_by_model) >= 2:
    paths = plot_model_comparison(metrics_by_model, LOCAL_PLOTS, REPORT_FIGURES)
    print(f"comparison figure: {paths['pdf']}")
else:
    print("Need at least 2 trained models for a comparison figure.")

## 6. Summary table

In [ ]:
import pandas as pd
rows = []
for m, met in metrics_by_model.items():
    rows.append({
        "model": m,
        "accuracy": met["accuracy"],
        "macro_precision": met["macro_precision"],
        "macro_recall": met["macro_recall"],
        "macro_f1": met["macro_f1"],
        "weighted_f1": met["weighted_f1"],
    })
df = pd.DataFrame(rows).set_index("model")
print(df.round(4).to_string())
# Save CSV for the report
out_csv = LOCAL_METRICS / "model_comparison.csv"
df.to_csv(out_csv)
print(f"\nsaved: {out_csv}")

## 7. Commit instructions

Commit the new files:
```bash
cd Image-Classification-with-CNN
git add results/plots/ report/figures/ results/metrics/
git commit -m "feat: evaluation figures and metrics"
git push origin main
```